## Transform Results Data
- Read bronze results table
- Keep only the columns required for analytics (drop url column)
- Standardise column names using snal case(driverId -> driver_id etc)
- Rename columns to make more meaningful (ex: date -> race_date)
- Filter rows where season,round,constructor_id or driver_id is null
- Remove duplicate records
- Transform values of columns race_name to Title Case
- Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.results'
silver_table = f'{catalog_name}.{silver_schema}.results'

###Step 1 - Read bronze circuits table

In [0]:
results_df = spark.table(bronze_table)
display(results_df)

### Step 2 - Keep only the columns required for analytics (drop url column)

In [0]:

from pyspark.sql import functions as F


In [0]:

results_dropped_df = results_df.drop('url')
display(results_dropped_df)

### Step 3
- Standardise column names using snal case(circuitId -> circuit_id)
- Rename columns to make them more meaningful(lat -> lattitude)

In [0]:
results_renamed_df = ( results_dropped_df
    .withColumnsRenamed(
        {
            'constructorId':'constructor_id',        'driverId':'driver_id',
            'raceName':'race_name',
            'date':'race_date',
            'grid':'grid_position',
            'laps':'completed_laps',
            'number':'car_number',
            'position':'final_position',
            'positionText':'final_position_text'
        }
        )
)


In [0]:
results_valid_df = ( results_renamed_df \
                    .filter( 
                         F.col("season").isNotNull() &
                         F.col("round").isNotNull() &
                         F.col('constructor_id').isNotNull() &
                         F.col('driver_id').isNotNull()    
                    ) )

In [0]:
display(results_valid_df.count() - results_renamed_df.count())

### Step 6 - Remove duplicate records

In [0]:

#USING dropDuplicates() METHOD
results_distinct_df = results_valid_df.dropDuplicates(["season","round","constructor_id","driver_id"])




### Step 7 -Transform values of columns circuit_name and locality to Title Case

In [0]:
results_final_df = (
    results_distinct_df
    .withColumn("race_name",F.initcap(F.col("race_name")))
    )
display(results_final_df)

### Step 8 - Write the transformed data to silver circuits table


In [0]:
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)

)

In [0]:
display(spark.table(silver_table))